In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
prompt = "Explain BM25 in simple words."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
!pip install -q peft trl datasets

In [ ]:
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2 datasets==2.19.1

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="qlora_training_data.json"
)

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
def format_example(example):
    return {
        "text": f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""
    }

dataset = dataset.map(format_example)

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tinyllama-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,

    fp16=False,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained("tinyllama-lora")
tokenizer.save_pretrained("tinyllama-lora")

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -q \
transformers==4.51.3 \
peft==0.15.2 \
trl==0.17.0 \
accelerate \
bitsandbytes \
sentencepiece

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cpu"
)

model = PeftModel.from_pretrained(
    base_model,
    "tinyllama-lora"
)

merged_model = model.merge_and_unload()

merged_model.save_pretrained("tinyllama-full")
tokenizer.save_pretrained("tinyllama-full")

In [ ]:
!pip uninstall -y transformers huggingface_hub
!pip install transformers==4.51.3 huggingface_hub==0.30.2

In [ ]:
from transformers import pipeline, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

pipe = pipeline(
    "text-generation",
    model="tinyllama-full",
    tokenizer=tokenizer,
    device_map="auto"
)

result = pipe(
    "Explain BM25 simply.",
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

print(result[0]["generated_text"])

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -r requirements.txt

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

tokenizer.save_pretrained("tinyllama-full")

In [ ]:
!cat tinyllama-full/tokenizer_config.json

In [ ]:
!pip install -q transformers==4.49.0 sentencepiece protobuf

In [ ]:
!rm -f /content/tinyllama-full/tokenizer*
!rm -f /content/tinyllama-full/special_tokens_map.json

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    use_fast=False
)

tokenizer.save_pretrained(
    "/content/tinyllama-full",
    legacy_format=True
)

In [ ]:
!cat /content/tinyllama-full/tokenizer_config.json

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/tinyllama-full \
    --outfile /content/tinyllama-f16.gguf \
    --outtype f16

In [ ]:
%cd /content/llama.cpp

!cmake -B build
!cmake --build build --config Release -j1

In [ ]:
!./build/bin/llama-quantize \
    /content/tinyllama-f16.gguf \
    /content/tinyllama-q4.gguf \
    q4_k_m

In [ ]:
from google.colab import files

files.download('/content/tinyllama-q4.gguf')